# Looped DiffusionBlocks LM — Experiments

Runs three ablation experiments on WikiText-103:
- **A: Baseline** — Standard 8-layer MLA LM, CE training
- **B: Looped DBlock** — Weight-tied 8L × K=4, DiffusionBlocks training
- **C: DBlock + Engram** — B + 1M-slot hash-based memory

Requires GPU runtime (A100 recommended).

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone repo and install dependencies
!git clone -b shortcut-distillation https://github.com/guyko81/DiffusionBlocks.git
%cd DiffusionBlocks
!pip install -q lightning wandb transformers datasets scipy

In [ ]:
# Configure batch size based on available GPU memory
import torch
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU memory: {gpu_mem_gb:.1f} GB")

if gpu_mem_gb >= 35:    # A100 40GB
    BATCH = 64
    ACCUM = 1
    GRAD_CKPT = ""
elif gpu_mem_gb >= 14:  # V100 16GB / T4 16GB
    BATCH = 32
    ACCUM = 2
    GRAD_CKPT = "--gradient_checkpointing"
else:                   # T4 etc
    BATCH = 16
    ACCUM = 4
    GRAD_CKPT = "--gradient_checkpointing"

print(f"Config: batch={BATCH}, accumulate={ACCUM}, grad_ckpt={bool(GRAD_CKPT)}")

In [ ]:
# Common training args
BASE = f"""python main_lm.py train \
    --seq_len 512 \
    --batch_size {BATCH} \
    --accumulate_grad_batches {ACCUM} \
    {GRAD_CKPT} \
    --lr 3e-4 \
    --weight_decay 0.1 \
    --warmup_steps 1000 \
    --scheduler_type cosine \
    --num_epochs 10 \
    --num_workers 2 \
    --precision bf16-mixed \
    --save_every_n_epochs 5 \
    --debug"""

## Experiment A: Baseline

In [ ]:
!{BASE} --model_variant baseline --postfix=-baseline

## Experiment B: Looped DBlock (K=4)

In [ ]:
!{BASE} --model_variant dblock --num_blocks 4 --postfix=-dblock-K4

## Experiment C: DBlock + Engram

In [ ]:
!{BASE} --model_variant dblock_engram --num_blocks 4 --engram_table_size 1000000 --postfix=-dblock-K4-engram

## Evaluation: Perplexity + Inference-Step Sweep

In [ ]:
import glob
import torch
from lm.config import LMConfig
from lm.model_lm import DBlockLM, BaselineLM
from lm.data_lm import WikiText103DataModule
from lm.eval_lm import compute_perplexity, sweep_inference_steps

# Setup data
dm = WikiText103DataModule(batch_size=32, seq_len=512, num_workers=2)
dm.setup()
test_dl = dm.test_dataloader()

# Find checkpoints
ckpts = sorted(glob.glob("logs/**/last.ckpt", recursive=True))
print("Found checkpoints:")
for c in ckpts:
    print(f"  {c}")

In [ ]:
# Evaluate baseline
baseline_ckpt = [c for c in ckpts if "baseline" in c][0]
baseline = BaselineLM.load_from_checkpoint(baseline_ckpt)
baseline = baseline.cuda().eval()
ppl = compute_perplexity(baseline, test_dl)
print(f"\nBaseline PPL: {ppl:.2f}")

In [ ]:
# Evaluate DBlock with inference-step sweep
dblock_ckpt = [c for c in ckpts if "dblock-K4" in c and "engram" not in c][0]
dblock = DBlockLM.load_from_checkpoint(dblock_ckpt)
dblock = dblock.cuda().eval()

print("DBlock inference-step sweep:")
dblock_results = sweep_inference_steps(dblock, test_dl, [1, 2, 3, 4])

In [ ]:
# Evaluate DBlock + Engram with inference-step sweep
engram_ckpt = [c for c in ckpts if "engram" in c][0]
engram_model = DBlockLM.load_from_checkpoint(engram_ckpt)
engram_model = engram_model.cuda().eval()

print("DBlock+Engram inference-step sweep:")
engram_results = sweep_inference_steps(engram_model, test_dl, [1, 2, 3, 4])

In [ ]:
# Summary table
print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"{'Model':<25} {'K':>3} {'PPL':>10}")
print("-"*40)
print(f"{'Baseline':<25} {'1':>3} {ppl:>10.2f}")
for k, p in dblock_results.items():
    print(f"{'DBlock':<25} {k:>3} {p:>10.2f}")
for k, p in engram_results.items():
    print(f"{'DBlock+Engram':<25} {k:>3} {p:>10.2f}")

In [ ]:
# Save checkpoints to Drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r logs /content/drive/MyDrive/dblocks-lm-logs